In [21]:
import os
import cv2
import json
import yaml
import math
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from pathlib import Path
from collections import Counter
from tqdm import tqdm

## 1) 경로 / 설정

In [22]:
import numpy as np
from pathlib import Path
import cv2
import random

DATA_ROOT = Path(r"../ai09-project01/sprint_ai_project1_data")
TRAIN_IMAGE_DIR = DATA_ROOT / "train_images"
TRAIN_ANNOTATION_DIR = DATA_ROOT / "train_annotations"

OUTPUT_ROOT = Path(r"../ai09-project01/sprint_ai_project1_data_removed")
OUT_IMAGE_DIR = OUTPUT_ROOT / "train_images"
OUT_ANNOTATION_DIR = OUTPUT_ROOT / "train_annotations"

# 원본 구조도 같이 유지할지
COPY_ORIGINALS = True

# 전체 이미지 중 몇 %에 대해 제거형 증강본을 추가 생성할지
AUGMENT_RATIO = 0.5

# 한 이미지에서 최대 몇 개 객체를 제거할지
MAX_REMOVE_PER_IMAGE = 1

# 다수 클래스만 우선 제거 대상으로 쓸지
USE_MAJORITY_CLASSES_ONLY = True

# 객체 수 상위 몇 % 클래스를 다수 클래스로 볼지
MAJORITY_TOP_PERCENT = 0.30

# bbox보다 조금 더 넓게 지우기
REMOVE_BOX_EXPAND_RATIO = 0.08

# inpainting 설정
INPAINT_RADIUS = 7
INPAINT_METHOD = cv2.INPAINT_TELEA  # 또는 cv2.INPAINT_NS

# 랜덤 시드
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 증강 파일명 suffix
AUG_SUFFIX = "_rm"

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("TRAIN_ANNOTATION_DIR:", TRAIN_ANNOTATION_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


TRAIN_IMAGE_DIR: ..\ai09-project01\sprint_ai_project1_data\train_images
TRAIN_ANNOTATION_DIR: ..\ai09-project01\sprint_ai_project1_data\train_annotations
OUTPUT_ROOT: ..\ai09-project01\sprint_ai_project1_data_removed


In [42]:
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

device = "cuda"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()

# 1660 SUPER 필수 옵션
pipe.enable_model_cpu_offload()

ModuleNotFoundError: No module named 'diffusers'

In [ ]:
def sd_inpaint(image_bgr, mask):
    # OpenCV(BGR) → PIL(RGB)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    image_pil = Image.fromarray(image_rgb)
    mask_pil = Image.fromarray(mask).convert("L")

    result = pipe(
        prompt="realistic background, clean surface",
        image=image_pil,
        mask_image=mask_pil,
        num_inference_steps=25,
        guidance_scale=7.5,
    ).images[0]

    result_np = np.array(result)
    result_bgr = cv2.cvtColor(result_np, cv2.COLOR_RGB2BGR)

    return result_bgr

## 2) baseline 노트북 기준 annotation 파싱 함수

In [23]:
def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default

def get_image_files(image_dir, exts=(".png", ".jpg", ".jpeg", ".bmp", ".webp")):
    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(exts)
    ])
    print("Number of image files:", len(image_files))
    print("Sample image files:", image_files[:5])
    return image_files

def build_images_df(image_dir, image_files):
    rows = []

    for file_name in image_files:
        img_path = os.path.join(image_dir, file_name)
        img = cv2.imread(img_path)

        if img is None:
            rows.append({
                "file_name": file_name,
                "width": None,
                "height": None,
                "channel": None,
                "is_broken": True
            })
            continue

        h, w, c = img.shape
        rows.append({
            "file_name": file_name,
            "width": w,
            "height": h,
            "channel": c,
            "is_broken": False
        })

    images_df = pd.DataFrame(rows)
    images_df["image_key"] = images_df["file_name"].apply(lambda x: os.path.splitext(x)[0])
    return images_df

def parse_annotation_json(json_path, image_folder_name=None):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    if isinstance(data, dict) and "annotations" in data:
        annotations = data.get("annotations", [])
        images = data.get("images", [])
        categories = data.get("categories", [])

        image_info = images[0] if len(images) > 0 else {}
        image_id = safe_get(image_info, ["id"], default=None)
        file_name = safe_get(image_info, ["file_name"], default=None)

        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in annotations:
            bbox = safe_get(ann, ["bbox"], default=None)
            category_id = safe_get(ann, ["category_id"], default=None)

            rows.append({
                "image_key": image_folder_name,
                "json_file": os.path.basename(json_path),
                "file_name_from_json": file_name,
                "image_id": image_id,
                "bbox": bbox,
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
                "dl_idx": safe_get(image_info, ["dl_idx"]),
                "dl_name": safe_get(image_info, ["dl_name"]),
                "drug_shape": safe_get(image_info, ["drug_shape"]),
                "color_class1": safe_get(image_info, ["color_class1"]),
                "color_class2": safe_get(image_info, ["color_class2"]),
                "line_front": safe_get(image_info, ["line_front"]),
                "line_back": safe_get(image_info, ["line_back"]),
                "print_front": safe_get(image_info, ["print_front"]),
                "print_back": safe_get(image_info, ["print_back"]),
                "json_path": json_path
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")

def build_annotations_df(annotation_root):
    rows = []

    for root, dirs, files in os.walk(annotation_root):
        json_files = [f for f in files if f.lower().endswith(".json")]
        if not json_files:
            continue

        # 핵심: JSON 파일명이 아니라 "상위 폴더명"을 image_key로 사용
        image_key = os.path.basename(root)

        rel_dir = os.path.relpath(root, annotation_root)
        dir_parts = rel_dir.split(os.sep)

        for jf in json_files:
            json_path = os.path.join(root, jf)

            try:
                parsed_rows = parse_annotation_json(json_path, image_folder_name=image_key)
                for row in parsed_rows:
                    row["relative_dir"] = rel_dir
                    row["dir_depth"] = len(dir_parts)
                    rows.append(row)
            except Exception as e:
                rows.append({
                    "image_key": image_key,
                    "json_file": jf,
                    "file_name_from_json": None,
                    "image_id": None,
                    "bbox": None,
                    "category_id": None,
                    "category_name": None,
                    "dl_idx": None,
                    "dl_name": None,
                    "json_path": json_path,
                    "relative_dir": rel_dir,
                    "dir_depth": len(dir_parts),
                    "parse_error": str(e)
                })

    annotations_df = pd.DataFrame(rows)
    print("annotations_df shape:", annotations_df.shape)
    return annotations_df

## 3) JSON 원본을 직접 읽어서 객체 제거용 메타정보 구성

In [24]:
from pathlib import Path

def find_json_files(annotation_root: Path):
    json_paths = []
    for root, dirs, files in os.walk(annotation_root):
        for f in files:
            if f.lower().endswith(".json"):
                json_paths.append(Path(root) / f)
    return sorted(json_paths)

def load_annotation_data(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def get_image_filename_from_annotation(data, fallback_stem=None):
    images = data.get("images", [])
    if len(images) > 0:
        file_name = images[0].get("file_name", None)
        if file_name not in [None, ""]:
            return os.path.basename(file_name)

    if fallback_stem is not None:
        # 이미지 확장자는 실제 파일에서 찾게 됨
        return None
    return None

def match_image_path(image_dir: Path, file_name=None, stem=None):
    candidates = []
    if file_name:
        p = image_dir / os.path.basename(file_name)
        if p.exists():
            return p

    if stem:
        for ext in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]:
            p = image_dir / f"{stem}{ext}"
            if p.exists():
                return p

    # 마지막 fallback: stem으로 시작하는 파일 찾기
    if stem:
        for p in image_dir.iterdir():
            if p.is_file() and p.stem == stem:
                return p
    return None

from collections import defaultdict
from copy import deepcopy
from pathlib import Path
from tqdm import tqdm
import os
import json

from collections import defaultdict
from copy import deepcopy
from pathlib import Path
from tqdm import tqdm
import os

def extract_dataset_records(image_dir: Path, annotation_dir: Path):
    """
    JSON 안의 images[0]['file_name'] 기준으로 이미지 단위 record 생성
    """
    grouped = defaultdict(list)

    json_paths = find_json_files(annotation_dir)

    # 1) 각 json을 읽고, 실제 이미지 파일명 기준으로 그룹화
    for json_path in tqdm(json_paths, desc="Grouping json files by true image file"):
        data = load_annotation_data(json_path)

        file_name = get_image_filename_from_annotation(
            data,
            fallback_stem=json_path.stem
        )
        if file_name is None:
            continue

        image_key = Path(file_name).stem   # 핵심
        grouped[image_key].append((json_path, data, file_name))

    records = []

    # 2) 같은 image_key에 속한 여러 json의 annotation을 합치기
    for image_key, items in tqdm(grouped.items(), desc="Loading grouped dataset records"):
        representative_json_path, representative_data, representative_file_name = items[0]

        img_path = match_image_path(
            image_dir,
            file_name=representative_file_name,
            stem=image_key
        )

        merged_annotations = []
        merged_annotation_dicts = []

        for json_path, data, file_name in items:
            anns = data.get("annotations", [])

            for ann in anns:
                bbox = ann.get("bbox", None)
                cat_id = ann.get("category_id", None)

                if isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                    merged_annotations.append({
                        "bbox": bbox,
                        "category_id": cat_id,
                        "annotation_id": ann.get("id", None),
                        "source_json": str(json_path)
                    })

                    ann_copy = deepcopy(ann)
                    ann_copy["_source_json"] = str(json_path)
                    ann_copy["_source_relpath"] = str(Path(json_path).relative_to(annotation_dir))
                    merged_annotation_dicts.append(ann_copy)

        merged_data = deepcopy(representative_data)
        merged_data["annotations"] = merged_annotation_dicts

        if "images" not in merged_data or not merged_data["images"]:
            merged_data["images"] = [{"file_name": representative_file_name}]
        else:
            merged_data["images"][0]["file_name"] = representative_file_name

        records.append({
            "image_key": image_key,
            "json_paths": [str(x[0]) for x in items],
            "image_path": img_path,
            "image_file_name": representative_file_name,
            "data": merged_data,
            "annotations": merged_annotations
        })

    return records

In [25]:
import pandas as pd

image_files = get_image_files(str(TRAIN_IMAGE_DIR))
images_df = build_images_df(str(TRAIN_IMAGE_DIR), image_files)
annotations_df = build_annotations_df(str(TRAIN_ANNOTATION_DIR))

records = extract_dataset_records(TRAIN_IMAGE_DIR, TRAIN_ANNOTATION_DIR)

print("총 record 수:", len(records))
print("첫 5개 image_key:", [r["image_key"] for r in records[:5]])

ann_counts = [len(r["annotations"]) for r in records]
print("annotation 개수 통계")
print("min:", min(ann_counts))
print("max:", max(ann_counts))
print("mean:", sum(ann_counts) / len(ann_counts))

from collections import Counter
print("annotation 개수 분포 상위:")
print(Counter(ann_counts).most_common(10))

# 이미지 매칭 실패 확인
unmatched = [r for r in records if r["image_path"] is None]
print("이미지 매칭 실패 수:", len(unmatched))
if len(unmatched) > 0:
    print("예시 unmatched JSON:", unmatched[0]["json_path"][:3])

Number of image files: 232
Sample image files: ['K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_75_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_90_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_75_000_200.png']
annotations_df shape: (763, 19)


Loading grouped dataset records: 100%|██████████| 232/232 [00:00<00:00, 3802.42it/s]

총 record 수: 232
첫 5개 image_key: ['K-001900-016548-019607-029451_0_2_0_2_70_000_200', 'K-001900-016548-019607-029451_0_2_0_2_75_000_200', 'K-001900-016548-019607-029451_0_2_0_2_90_000_200', 'K-001900-016548-019607-033009_0_2_0_2_70_000_200', 'K-001900-016548-019607-033009_0_2_0_2_75_000_200']
annotation 개수 통계
min: 2
max: 4
mean: 3.288793103448276
annotation 개수 분포 상위:
[(3, 151), (4, 74), (2, 7)]
이미지 매칭 실패 수: 0


In [26]:
ann_count_per_image = annotations_df.groupby("image_key").size().sort_values()
print(ann_count_per_image.head(20))
print(ann_count_per_image.describe())

image_key
K-027993    3
K-022362    3
K-021771    3
K-029451    3
K-019552    3
K-027926    3
K-024850    3
K-031885    3
K-033009    3
K-013395    3
K-016551    3
K-012247    3
K-034597    3
K-012081    3
K-033208    3
K-027777    3
K-003743    3
K-004543    5
K-029345    6
K-019861    6
dtype: int64
count     56.000000
mean      13.625000
std       20.823555
min        3.000000
25%        3.000000
50%        9.000000
75%       16.000000
max      153.000000
dtype: float64


## 4) 클래스 분포 계산 + 제거 대상 클래스 선택

In [31]:
class_counter = Counter()

for r in records:
    for ann in r["annotations"]:
        if ann["category_id"] is not None:
            class_counter[ann["category_id"]] += 1

print("클래스별 객체 수 상위 20개:")
for cls_id, cnt in class_counter.most_common(20):
    print(f"class {cls_id}: {cnt}")

sorted_classes = [cls for cls, _ in class_counter.most_common()]
top_k = max(1, int(len(sorted_classes) * MAJORITY_TOP_PERCENT))
majority_classes = set(sorted_classes[:top_k])

print("\n다수 클래스로 간주할 category_id 목록:")
print(sorted(list(majority_classes)))

[{'image_key': 'K-001900-016548-019607-029451_0_2_0_2_70_000_200', 'json_paths': ['..\\ai09-project01\\sprint_ai_project1_data\\train_annotations\\K-001900-016548-019607-029451_json\\K-001900\\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json', '..\\ai09-project01\\sprint_ai_project1_data\\train_annotations\\K-001900-016548-019607-029451_json\\K-016548\\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json', '..\\ai09-project01\\sprint_ai_project1_data\\train_annotations\\K-001900-016548-019607-029451_json\\K-019607\\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json', '..\\ai09-project01\\sprint_ai_project1_data\\train_annotations\\K-001900-016548-019607-029451_json\\K-029451\\K-001900-016548-019607-029451_0_2_0_2_70_000_200.json'], 'image_path': WindowsPath('../ai09-project01/sprint_ai_project1_data/train_images/K-001900-016548-019607-029451_0_2_0_2_70_000_200.png'), 'image_file_name': 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'data': {'images': [{'file_name': 'K

## 5) 객체 제거(inpainting) + annotation JSON 갱신 함수

In [28]:
def xywh_to_xyxy(bbox, img_w, img_h):
    x, y, w, h = bbox
    x1 = max(0, min(int(round(x)), img_w - 1))
    y1 = max(0, min(int(round(y)), img_h - 1))
    x2 = max(0, min(int(round(x + w)), img_w - 1))
    y2 = max(0, min(int(round(y + h)), img_h - 1))
    return x1, y1, x2, y2

def expand_box(x1, y1, x2, y2, img_w, img_h, expand_ratio=0.08):
    bw = x2 - x1
    bh = y2 - y1
    pad_x = int(bw * expand_ratio)
    pad_y = int(bh * expand_ratio)
    nx1 = max(0, x1 - pad_x)
    ny1 = max(0, y1 - pad_y)
    nx2 = min(img_w - 1, x2 + pad_x)
    ny2 = min(img_h - 1, y2 + pad_y)
    return nx1, ny1, nx2, ny2

def choose_annotations_to_remove(annotations, majority_classes, max_remove=1, majority_only=True):
    candidate_indices = []
    for i, ann in enumerate(annotations):
        cls_id = ann.get("category_id", None)
        if majority_only:
            if cls_id in majority_classes:
                candidate_indices.append(i)
        else:
            candidate_indices.append(i)

    if len(candidate_indices) == 0:
        return []

    remove_count = random.randint(1, min(max_remove, len(candidate_indices)))
    return sorted(random.sample(candidate_indices, remove_count))

def build_inpaint_mask_from_annotations(annotations, remove_indices, img_h, img_w, expand_ratio=0.08):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)

    for idx in remove_indices:
        bbox = annotations[idx]["bbox"]
        x1, y1, x2, y2 = xywh_to_xyxy(bbox, img_w, img_h)
        x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, img_w, img_h, expand_ratio)

        mask[y1:y2, x1:x2] = 255

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        ax = max(1, (x2 - x1) // 2)
        ay = max(1, (y2 - y1) // 2)

        ellipse_mask = np.zeros_like(mask)
        cv2.ellipse(ellipse_mask, (cx, cy), (ax, ay), 0, 0, 360, 255, -1)
        mask = np.maximum(mask, ellipse_mask)

    return mask

def remove_objects_with_inpaint(image, mask, radius=7, method=cv2.INPAINT_TELEA):
    return cv2.inpaint(image, mask, radius, method)

def update_annotation_json_data(original_data, remove_indices):
    new_data = deepcopy(original_data)
    anns = new_data.get("annotations", [])
    new_anns = [ann for i, ann in enumerate(anns) if i not in remove_indices]
    new_data["annotations"] = new_anns
    return new_data

from copy import deepcopy
from pathlib import Path
import cv2
import json
import numpy as np

def build_mask_from_annotations(annotations, remove_indices, image_shape, expand_ratio=0.08):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    for idx in remove_indices:
        ann = annotations[idx]
        x, y, bw, bh = ann["bbox"]  # COCO xywh
        x1 = max(0, int(x - bw * expand_ratio))
        y1 = max(0, int(y - bh * expand_ratio))
        x2 = min(w - 1, int(x + bw + bw * expand_ratio))
        y2 = min(h - 1, int(y + bh + bh * expand_ratio))

        mask[y1:y2, x1:x2] = 255

    return mask

def choose_annotations_to_remove(annotations, majority_classes=None, max_remove=1, majority_only=True):
    removable = []

    for i, ann in enumerate(annotations):
        cid = ann.get("category_id", None)
        if majority_only:
            if cid in majority_classes:
                removable.append(i)
        else:
            removable.append(i)

    if len(removable) == 0:
        return []

    remove_n = min(max_remove, len(removable))
    remove_n = max(1, remove_n)
    return random.sample(removable, remove_n)

def save_augmented_record(
    record,
    out_img_dir,
    out_ann_dir,
    majority_classes,
    max_remove=1,
    majority_only=True,
    expand_ratio=0.08,
    inpaint_radius=7,
    inpaint_method=cv2.INPAINT_TELEA,
    aug_suffix="_rm"
):
    if record["image_path"] is None or not Path(record["image_path"]).exists():
        return False, "image_not_found"

    image = cv2.imread(str(record["image_path"]))
    if image is None:
        return False, "image_read_fail"

    annotations = deepcopy(record["data"].get("annotations", []))
    if len(annotations) == 0:
        return False, "no_annotations"

    remove_indices = choose_annotations_to_remove(
        annotations=annotations,
        majority_classes=majority_classes,
        max_remove=max_remove,
        majority_only=majority_only
    )

    if len(remove_indices) == 0:
        return False, "no_removable_annotations"

    if len(remove_indices) >= len(annotations):
        return False, "all_annotations_removed"

    # 이미지 inpainting
    mask = build_mask_from_annotations(
        annotations=annotations,
        remove_indices=remove_indices,
        image_shape=image.shape,
        expand_ratio=expand_ratio
    )
    # aug_image = cv2.inpaint(image, mask, inpaint_radius, inpaint_method)
    mask = cv2.GaussianBlur(mask, (15, 15), 0)

    aug_image = sd_inpaint(image, mask)

    # 제거 후 남길 annotation
    kept_annotations = [ann for i, ann in enumerate(annotations) if i not in remove_indices]

    # 새 이미지 저장
    orig_name = Path(record["image_file_name"]).name
    orig_stem = Path(orig_name).stem
    orig_suffix = Path(orig_name).suffix

    new_img_name = f"{orig_stem}{aug_suffix}{orig_suffix}"
    out_img_path = Path(out_img_dir) / new_img_name
    cv2.imwrite(str(out_img_path), aug_image)

    # 새 annotation 루트: [이미지명_json] 폴더
    image_json_root = Path(out_ann_dir) / f"{orig_stem}{aug_suffix}_json"
    image_json_root.mkdir(parents=True, exist_ok=True)

    # kept annotation들을 원래 source json별로 다시 묶기
    grouped_by_source = {}
    for ann in kept_annotations:
        src_rel = ann.get("_source_relpath", None)
        src_json = ann.get("_source_json", None)

        if src_rel is None and src_json is not None:
            src_rel = Path(src_json).name

        if src_rel is None:
            src_rel = "unknown/ann.json"

        grouped_by_source.setdefault(src_rel, []).append(ann)

    # 원래 json별로 저장
    for src_rel, anns_in_one_json in grouped_by_source.items():
        src_rel = Path(src_rel)

        # 예: IMGNAME/pill_001/ann.json 이면
        # 맨 앞의 이미지명 폴더는 버리고 그 아래 구조만 유지
        rel_parts = src_rel.parts
        if len(rel_parts) >= 2:
            relative_inside_image_json = Path(*rel_parts[1:])
        else:
            relative_inside_image_json = Path(src_rel.name)

        out_json_path = image_json_root / relative_inside_image_json
        out_json_path.parent.mkdir(parents=True, exist_ok=True)

        # 대표 data 뼈대
        json_data = {
            "images": deepcopy(record["data"].get("images", [])),
            "annotations": [],
            "categories": deepcopy(record["data"].get("categories", []))
        }

        if not json_data["images"]:
            json_data["images"] = [{"file_name": new_img_name}]
        else:
            json_data["images"][0]["file_name"] = new_img_name

        # source 메타 제거하고 annotation 넣기
        cleaned_anns = []
        for ann in anns_in_one_json:
            ann_copy = deepcopy(ann)
            ann_copy.pop("_source_json", None)
            ann_copy.pop("_source_relpath", None)
            cleaned_anns.append(ann_copy)

        json_data["annotations"] = cleaned_anns

        with open(out_json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)

    return True, {
        "removed_count": len(remove_indices),
        "kept_count": len(kept_annotations),
        "saved_image": str(out_img_path),
        "saved_json_root": str(image_json_root)
    }

## 6) 원본 복사 + 제거형 증강 데이터 생성

In [29]:
from pathlib import Path

missing = []

for r in records:
    for jp in r["json_paths"]:
        if not Path(jp).exists():
            missing.append(jp)

print("missing json files:", len(missing))
print("sample:", missing[:5])

missing json files: 0
sample: []


In [38]:
import shutil
from collections import Counter
import random
from tqdm import tqdm
from pathlib import Path

# =========================
# OUTPUT 폴더 초기화
# =========================
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# records 유효성 필터링
# =========================
valid_records = []

for r in records:
    if r.get("image_path") is None:
        continue
    if not Path(r["image_path"]).exists():
        continue
    if "annotations" not in r or len(r["annotations"]) == 0:
        continue

    valid_records.append(r)

print(f"전체 records: {len(records)}")
print(f"유효한 records: {len(valid_records)}")


# =========================
# 원본 복사
# =========================
if COPY_ORIGINALS:
    for r in tqdm(valid_records, desc="Copying originals"):
        shutil.copy2(r["image_path"], OUT_IMAGE_DIR / Path(r["image_path"]).name)

        ann_subdir = OUT_ANNOTATION_DIR / r["image_key"]
        ann_subdir.mkdir(parents=True, exist_ok=True)


# =========================
# 증강 대상 선택
# =========================
num_to_augment = int(len(valid_records) * AUGMENT_RATIO)

if num_to_augment == 0:
    raise ValueError("AUGMENT_RATIO가 너무 작아서 선택된 데이터가 없음")

selected_records = random.sample(valid_records, num_to_augment)


# =========================
# 증강 수행
# =========================
success_count = 0
fail_stats = Counter()

for r in tqdm(selected_records, desc="Generating remove-mask augmented samples"):
    try:
        ok, info = save_augmented_record(
            record=r,
            out_img_dir=OUT_IMAGE_DIR,
            out_ann_dir=OUT_ANNOTATION_DIR,
            majority_classes=majority_classes,
            max_remove=MAX_REMOVE_PER_IMAGE,
            majority_only=USE_MAJORITY_CLASSES_ONLY
        )

        if ok:
            success_count += 1
        else:
            fail_stats[info] += 1

    except Exception as e:
        fail_stats[str(e)] += 1


# =========================
# 결과 출력
# =========================
print("\n원본 record 수:", len(records))
print("유효 record 수:", len(valid_records))
print("증강 시도 수:", len(selected_records))
print("증강 성공 수:", success_count)
print("실패 통계:", dict(fail_stats))
print("출력 폴더:", OUTPUT_ROOT)

전체 records: 232
유효한 records: 232


Generating remove-mask augmented samples: 100%|██████████| 116/116 [00:28<00:00,  4.07it/s]


원본 record 수: 232
유효 record 수: 232
증강 시도 수: 116
증강 성공 수: 0
실패 통계: {"[Errno 2] No such file or directory: '..\\\\ai09-project01\\\\sprint_ai_project1_data_removed\\\\train_annotations\\\\K-003351-031863-038162_0_2_0_2_70_000_200_rm_json\\\\K-003351\\\\K-003351-031863-038162_0_2_0_2_70_000_200.json'": 1, "[Errno 2] No such file or directory: '..\\\\ai09-project01\\\\sprint_ai_project1_data_removed\\\\train_annotations\\\\K-003351-003832-016262_0_2_0_2_90_000_200_rm_json\\\\K-003351\\\\K-003351-003832-016262_0_2_0_2_90_000_200.json'": 1, "[Errno 2] No such file or directory: '..\\\\ai09-project01\\\\sprint_ai_project1_data_removed\\\\train_annotations\\\\K-001900-016548-021771-027926_0_2_0_2_70_000_200_rm_json\\\\K-001900\\\\K-001900-016548-021771-027926_0_2_0_2_70_000_200.json'": 1, "[Errno 2] No such file or directory: '..\\\\ai09-project01\\\\sprint_ai_project1_data_removed\\\\train_annotations\\\\K-003483-016232-022347-025469_0_2_0_2_90_000_200_rm_json\\\\K-016232\\\\K-003483-016232-0

## 7) 전/후 시각화

In [40]:
def draw_boxes_from_annotations(image, annotations, color=(0, 255, 0), thickness=2):
    canvas = image.copy()
    h, w = canvas.shape[:2]
    for ann in annotations:
        bbox = ann.get("bbox", None)
        cls_id = ann.get("category_id", None)
        if not (isinstance(bbox, list) and len(bbox) == 4):
            continue
        x1, y1, x2, y2 = xywh_to_xyxy(bbox, w, h)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(canvas, f"{cls_id}", (x1, max(20, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return canvas

def show_before_after_by_stem(json_stem):
    orig_json = TRAIN_ANNOTATION_DIR / f"{json_stem}.json"
    aug_json = OUT_ANNOTATION_DIR / f"{json_stem}{AUG_SUFFIX}.json"

    if not orig_json.exists() or not aug_json.exists():
        print("해당 원본/증강 JSON을 찾지 못했어.")
        return

    with open(orig_json, "r", encoding="utf-8") as f:
        orig_data = json.load(f)
    with open(aug_json, "r", encoding="utf-8") as f:
        aug_data = json.load(f)

    orig_file_name = orig_data["images"][0]["file_name"]
    aug_file_name = aug_data["images"][0]["file_name"]

    orig_img_path = TRAIN_IMAGE_DIR / orig_file_name
    aug_img_path = OUT_IMAGE_DIR / aug_file_name

    orig_img = cv2.cvtColor(cv2.imread(str(orig_img_path)), cv2.COLOR_BGR2RGB)
    aug_img = cv2.cvtColor(cv2.imread(str(aug_img_path)), cv2.COLOR_BGR2RGB)

    orig_vis = draw_boxes_from_annotations(orig_img, orig_data.get("annotations", []), color=(0,255,0))
    aug_vis = draw_boxes_from_annotations(aug_img, aug_data.get("annotations", []), color=(255,0,0))

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(orig_vis)
    axes[0].set_title(f"Original: {orig_file_name}")
    axes[0].axis("off")

    axes[1].imshow(aug_vis)
    axes[1].set_title(f"Augmented: {aug_file_name}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    print("Original annotation count:", len(orig_data.get("annotations", [])))
    print("Augmented annotation count:", len(aug_data.get("annotations", [])))

In [41]:

# 예시: 실제 stem 이름으로 바꿔서 실행
show_before_after_by_stem("sample_name_without_ext")


해당 원본/증강 JSON을 찾지 못했어.


## 8) 필요하면 YOLO 변환용 후속 처리에 연결

이 노트북은 **원본 JSON annotation 구조를 유지한 증강 데이터셋**을 생성한다.
이후 RT-DETRv4 학습 전처리 또는 기존 baseline의 COCO/YOLO 변환 코드에 이 출력 폴더를 입력으로 넣어서 이어서 사용할 수 있다.